# Phase 2 - Try Resizing a Running Job

Two tests:

1. **Through Kubeflow Trainer** - start a TrainJob, try to patch `numNodes`.
   Expect it to be rejected. The error message is the deliverable.

2. **Through JobSet directly** - enable `ElasticJobSet` feature gate, start a
   job on JobSet, change parallelism up and down. If JobSet works but Trainer
   doesn't, that proves the gap is in the Trainer layer.

## Setup

In [ ]:
!pip install --force-reinstall --no-cache-dir -U \
    "kubeflow @ git+https://github.com/opendatahub-io/kubeflow-sdk.git@v0.3.0+rhaiv.2"
!pip install yamlmagic hatchling --index-url https://pypi.org/simple
!pip install --no-deps .. --index-url https://pypi.org/simple
%load_ext yamlmagic

In [ ]:
%%yaml parameters

namespace: elastic-scaling
duration_minutes: 10

In [ ]:
%load_ext autoreload
%autoreload 2

from elastic_scaling_poc.phase2.resize_test import train_func

print("train_func loaded")

In [ ]:
import os
from pathlib import Path

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubernetes import client as k8s

api_server = os.environ["OPENSHIFT_API_URL"]
token = os.getenv("NOTEBOOK_USER_TOKEN", "")
if not token:
    sa_path = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
    if sa_path.exists():
        token = sa_path.read_text().strip()
if not token:
    raise RuntimeError(
        "Set NOTEBOOK_USER_TOKEN, or run inside a workbench "
        "with a service-account token."
    )

config = k8s.Configuration()
config.host = api_server
config.api_key = {"authorization": f"Bearer {token}"}
config.verify_ssl = False

client = TrainerClient(
    KubernetesBackendConfig(
        namespace=parameters["namespace"],
        client_configuration=config,
    )
)

## Part 1 - Resize through Kubeflow Trainer

Submit a TrainJob on 1 GPU, wait for it to start, then try to patch
`numNodes` from 1 to 2. The research doc predicts two blockers:

1. `TrainJob.spec.trainer` is immutable - the patch is refused before
   it even reaches JobSet.
2. Even if it did reach JobSet, the `PET_NNODES` env var change would
   trigger the webhook rejection.

The error message itself is the deliverable for this part.

In [ ]:
from kubeflow.trainer.options import Name
from kubeflow.trainer.rhai import TransformersTrainer

trainer = TransformersTrainer(
    func=train_func,
    func_args=parameters,
    num_nodes=1,
    resources_per_node={"nvidia.com/gpu": 1},
)

runtime = client.backend.get_runtime("torch-distributed")
JOB_NAME = client.train(
    trainer=trainer,
    runtime=runtime,
    options=[Name("phase2-resize-trainer")],
)
print(f"Submitted: {JOB_NAME}")

In [ ]:
import time

# Wait for the job to start running
for _ in range(60):
    status = client.get_job(JOB_NAME).status
    print(f"Status: {status}")
    if status == "Running":
        break
    time.sleep(10)

### Try to patch numNodes

This should fail. Capture the exact error.

In [ ]:
!oc patch trainjob phase2-resize-trainer -n {parameters["namespace"]} \
    --type=merge -p '{"spec": {"trainer": {"numNodes": 2}}}'

In [ ]:
# Cleanup Part 1
client.delete_job(name=JOB_NAME)
print(f"Deleted {JOB_NAME}")

## Part 2 - Resize through JobSet directly

Skip Kubeflow Trainer entirely. Create a JobSet with `completionMode: Indexed`
and the `ElasticJobSet` feature gate enabled on the controller.

If this works but Part 1 doesn't, it proves the gap is specifically in the
Trainer layer (immutable spec + PET_NNODES coupling).

**Prerequisite:** ElasticJobSet feature gate must be enabled on the JobSet
controller. See the research doc section 8 for how to do this.

In [ ]:
# Verify the feature gate is active
!oc logs deploy/jobset-controller-manager -n openshift-jobset-operator 2>&1 | grep -i elastic

In [ ]:
%%bash
cat <<'EOF' | oc apply -f -
apiVersion: jobset.x-k8s.io/v1alpha2
kind: JobSet
metadata:
  name: phase2-resize-jobset
  namespace: elastic-scaling
spec:
  replicatedJobs:
    - name: workers
      replicas: 1
      template:
        spec:
          completionMode: Indexed
          completions: 1
          parallelism: 1
          template:
            spec:
              containers:
                - name: worker
                  image: docker.io/pytorch/pytorch:2.5.1-cuda12.4-cudnn9-runtime
                  command: ["python", "-c"]
                  args:
                    - |
                      import time
                      print("JobSet worker running", flush=True)
                      for i in range(600):
                          time.sleep(1)
                          if i % 60 == 0:
                              print(f"minute {i//60}/10", flush=True)
                  resources:
                    limits:
                      nvidia.com/gpu: "1"
                    requests:
                      nvidia.com/gpu: "1"
              restartPolicy: OnFailure
EOF

In [ ]:
# Wait for it to start
!oc get jobset phase2-resize-jobset -n {parameters["namespace"]} -w

### Try to scale up: parallelism 1 -> 2

In [ ]:
!oc patch jobset phase2-resize-jobset -n {parameters["namespace"]} \
    --type=json -p '[{"op": "replace", "path": "/spec/replicatedJobs/0/template/spec/parallelism", "value": 2}, {"op": "replace", "path": "/spec/replicatedJobs/0/template/spec/completions", "value": 2}]'

In [ ]:
# Check status after scale-up attempt
!oc get jobset phase2-resize-jobset -n {parameters["namespace"]} -o yaml | grep -A 5 'parallelism\|completions\|status'

### Try to scale down: parallelism 2 -> 1

In [ ]:
!oc patch jobset phase2-resize-jobset -n {parameters["namespace"]} \
    --type=json -p '[{"op": "replace", "path": "/spec/replicatedJobs/0/template/spec/parallelism", "value": 1}, {"op": "replace", "path": "/spec/replicatedJobs/0/template/spec/completions", "value": 1}]'

In [ ]:
!oc get jobset phase2-resize-jobset -n {parameters["namespace"]} -o yaml | grep -A 5 'parallelism\|completions\|status'

## Cleanup

In [ ]:
!oc delete jobset phase2-resize-jobset -n {parameters["namespace"]} --ignore-not-found
!oc delete trainjob phase2-resize-trainer -n {parameters["namespace"]} --ignore-not-found

## Results

Record what happened:

| Test | Expected | Actual | Error message |
|------|----------|--------|---------------|
| Patch TrainJob numNodes | Rejected (immutable spec) | | |
| Scale up JobSet parallelism | Accepted (ElasticJobSet) | | |
| Scale down JobSet parallelism | Accepted (ElasticJobSet) | | |

If Part 1 fails and Part 2 succeeds, the gap is confirmed: the blocker is
in the Kubeflow Trainer layer, not in JobSet.